In [1]:
!pip install -U transformers accelerate bitsandbytes sentence-transformers faiss-cpu

In [2]:
import sqlite3
import faiss
import numpy as np
import torch
import re

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
conn = sqlite3.connect("data.db")
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS users")
cursor.execute("DROP TABLE IF EXISTS orders")

cursor.execute("""
CREATE TABLE users (
    id INTEGER PRIMARY KEY,
    name TEXT,
    age INTEGER,
    country TEXT
)
""")

cursor.execute("""
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    user_id INTEGER,
    amount INTEGER
)
""")

cursor.executemany("""
INSERT INTO users (name, age, country) VALUES (?, ?, ?)
""", [
    ("Alice", 25, "USA"),
    ("Bob", 30, "India"),
    ("Charlie", 35, "UK")
])

cursor.executemany("""
INSERT INTO orders (user_id, amount) VALUES (?, ?)
""", [
    (1, 100),
    (2, 200),
    (3, 300)
])

conn.commit()
conn.close()

print("Database ready ✅")

Database ready ✅


In [4]:
schema_chunks = [
    "Table users has columns id, name, age, country",
    "Table orders has columns order_id, user_id, amount and is linked to users via user_id"
]

In [5]:
embed_model = SentenceTransformer('all-MiniLM-L6-v2')

embeddings = embed_model.encode(schema_chunks)

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings))

print("FAISS ready ✅")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

FAISS ready ✅


In [6]:
def retrieve_schema(query):
    query_embedding = embed_model.encode([query])
    distances, indices = index.search(np.array(query_embedding), k=2)
    results = [schema_chunks[i] for i in indices[0]]
    return "\n".join(results)

In [7]:
from transformers import BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    bnb_4bit_compute_dtype=torch.float16
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config
)

print("Model loaded ✅")

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded ✅


In [8]:
def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
def generate_sql(query):
    relevant_schema = retrieve_schema(query)

    prompt = f"""<s>[INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
{relevant_schema}

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

{query}

Only output SQL.
[/INST]"""

    return generate_text(prompt)

In [10]:
def extract_sql(text):
    import re

    matches = re.findall(r"(SELECT .*?;)", text, re.IGNORECASE | re.DOTALL)

    if matches:
        return matches[-1]   # 🔥 TAKE LAST MATCH

    return None

In [11]:
def run_sql(sql):
    conn = sqlite3.connect("data.db")
    cursor = conn.cursor()

    cursor.execute(sql)
    results = cursor.fetchall()

    conn.close()
    return results

In [12]:
def safe_generate_sql(query):
    raw_output = generate_sql(query)
    print("\nRAW OUTPUT:\n", raw_output)

    sql = extract_sql(raw_output)
    return sql

In [13]:
test_cases = [
    {"query": "Show all users older than 25", "expected_sql": "SELECT * FROM users WHERE age > 25;"},
    {"query": "List all users", "expected_sql": "SELECT * FROM users;"},
    {"query": "Show users from India", "expected_sql": "SELECT * FROM users WHERE country = 'India';"},
    {"query": "List all orders", "expected_sql": "SELECT * FROM orders;"},
    {"query": "Show orders with amount greater than 150", "expected_sql": "SELECT * FROM orders WHERE amount > 150;"},
    {"query": "Show all orders with user names", "expected_sql": "SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;"}
]

In [14]:
def evaluate_system(test_cases):
    correct = 0
    total = len(test_cases)

    for test in test_cases:
        query = test["query"]
        expected = test["expected_sql"]

        predicted = safe_generate_sql(query)

        print("\nQuery:", query)
        print("Expected:", expected)
        print("Predicted:", predicted)

        if predicted and predicted.strip().lower() == expected.strip().lower():
            correct += 1

    accuracy = correct / total
    print(f"\nAccuracy: {accuracy * 100:.2f}%")

In [15]:
evaluate_system(test_cases)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



RAW OUTPUT:
 [INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
Table users has columns id, name, age, country
Table orders has columns order_id, user_id, amount and is linked to users via user_id

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

Show all users older than 25

Only output SQL.
[/INST] SELECT * FROM users WHERE age > 25;

Query: Show all users older than 25
Expected: SELECT * FROM users WHERE age > 25;
Predicted: SELECT * FROM users WHERE age > 25;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



RAW OUTPUT:
 [INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
Table users has columns id, name, age, country
Table orders has columns order_id, user_id, amount and is linked to users via user_id

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

List all users

Only output SQL.
[/INST] SELECT * FROM users;

Query: List all users
Expected: SELECT * FROM users;
Predicted: SELECT * FROM users;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



RAW OUTPUT:
 [INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
Table users has columns id, name, age, country
Table orders has columns order_id, user_id, amount and is linked to users via user_id

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

Show users from India

Only output SQL.
[/INST] SELECT * FROM users WHERE country = 'India';

Query: Show users from India
Expected: SELECT * FROM users WHERE country = 'India';
Predicted: SELECT * FROM users WHERE country = 'India';


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



RAW OUTPUT:
 [INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
Table orders has columns order_id, user_id, amount and is linked to users via user_id
Table users has columns id, name, age, country

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

List all orders

Only output SQL.
[/INST] SELECT * FROM orders;

Query: List all orders
Expected: SELECT * FROM orders;
Predicted: SELECT * FROM orders;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



RAW OUTPUT:
 [INST]
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
Table orders has columns order_id, user_id, amount and is linked to users via user_id
Table users has columns id, name, age, country

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

Show orders with amount greater than 150

Only output SQL.
[/INST] SELECT * FROM orders WHERE amount > 150;

Query: Show orders with amount greater than 150
Expected: SELECT * FROM orders WHERE amount > 150;
Predicted: SELECT * FROM orders WHERE amount > 150;

RAW OUTPUT:
 [INST]
You are an expert SQL genera

In [16]:
# LANGCHAIN INTEGRATION (NEW)
!pip install langchain langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.6 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [16]:
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline

hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=False
)

llm = HuggingFacePipeline(pipeline=hf_pipeline)

print("LangChain LLM ready ✅")

/tmp/ipykernel_15234/1610181748.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline
[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


LangChain LLM ready ✅


/tmp/ipykernel_15234/1610181748.py:12: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=hf_pipeline)


In [17]:
from langchain_core.prompts import PromptTemplate

prompt_template = PromptTemplate(
    input_variables=["schema", "query"],
    template="""
You are an expert SQL generator.

Given the schema, generate correct SQL.

Schema:
{schema}

Examples:
- Show all users older than 25 → SELECT * FROM users WHERE age > 25;
- List all users → SELECT * FROM users;
- Show users from India → SELECT * FROM users WHERE country = 'India';
- Show all orders → SELECT * FROM orders;
- Show orders with amount > 150 → SELECT * FROM orders WHERE amount > 150;
- Show all orders with user names → SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

Now generate SQL for:

{query}

Only output SQL.
"""
)

print("Prompt template ready ✅")

Prompt template ready ✅


In [18]:

chain = prompt_template | llm

print("LangChain created ✅")

LangChain created ✅


In [28]:
def generate_sql_langchain(query):
    schema = retrieve_schema(query)

    response = chain.invoke({
        "schema": schema,
        "query": query
    })

    sql = extract_sql(response)

    return sql

In [29]:
def safe_generate_sql_langchain(query):
    sql = generate_sql_langchain(query)

    return sql

In [30]:
def evaluate_langchain(test_cases):
    correct = 0
    total = len(test_cases)

    for test in test_cases:
        query = test["query"]
        expected = test["expected_sql"]

        predicted = safe_generate_sql_langchain(query)

        print("\nQuery:", query)
        print("Expected:", expected)
        print("Predicted:", predicted)

        if predicted and predicted.strip().lower() == expected.strip().lower():
            correct += 1

    accuracy = correct / total
    print(f"\nLangChain Accuracy: {accuracy * 100:.2f}%")

In [32]:
evaluate_langchain(test_cases)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: Show all users older than 25
Expected: SELECT * FROM users WHERE age > 25;
Predicted: SELECT * FROM users WHERE age > 25;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: List all users
Expected: SELECT * FROM users;
Predicted: SELECT * FROM users;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: Show users from India
Expected: SELECT * FROM users WHERE country = 'India';
Predicted: SELECT * FROM users WHERE country = 'India';


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: List all orders
Expected: SELECT * FROM orders;
Predicted: SELECT * FROM orders;


[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Query: Show orders with amount greater than 150
Expected: SELECT * FROM orders WHERE amount > 150;
Predicted: SELECT * FROM orders WHERE amount > 150;

Query: Show all orders with user names
Expected: SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;
Predicted: SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;

LangChain Accuracy: 100.00%


In [33]:
def validate_sql(sql):

    if sql is None:
        return False

    sql_lower = sql.lower()

    forbidden = [
        "drop",
        "delete",
        "truncate",
        "update",
        "insert"
    ]

    for word in forbidden:
        if word in sql_lower:
            return False

    return sql_lower.startswith("select")

In [34]:
print(validate_sql("SELECT * FROM users;"))
print(validate_sql("DROP TABLE users;"))

True
False


In [35]:
failure_log = []

In [36]:
def log_failure(query, expected, predicted, reason):

    failure_log.append({
        "query": query,
        "expected": expected,
        "predicted": predicted,
        "reason": reason
    })

In [37]:
log_failure(
    query="Show all users",
    expected="SELECT * FROM users;",
    predicted="SELECT * FROM orders;",
    reason="wrong_table"
)

print(failure_log)

[{'query': 'Show all users', 'expected': 'SELECT * FROM users;', 'predicted': 'SELECT * FROM orders;', 'reason': 'wrong_table'}]


In [38]:
def classify_failure(expected, predicted):

    if predicted is None:
        return "generation_failure"

    expected = expected.lower()
    predicted = predicted.lower()

    # Missing JOIN
    if "join" in expected and "join" not in predicted:
        return "missing_join"

    # Wrong table
    if "users" in expected and "orders" in predicted:
        return "wrong_table"

    if "orders" in expected and "users" in predicted and "join" not in expected:
        return "wrong_table"

    # Exact mismatch
    if expected != predicted:
        return "sql_mismatch"

    return "correct"

In [39]:
print(
    classify_failure(
        "SELECT * FROM users;",
        "SELECT * FROM orders;"
    )
)

wrong_table


In [40]:
print(
    classify_failure(
        "SELECT users.name, orders.amount FROM users JOIN orders ON users.id = orders.user_id;",
        "SELECT * FROM orders;"
    )
)

missing_join


In [41]:
print(
    classify_failure(
        "SELECT * FROM users;",
        "SELECT * FROM users;"
    )
)

correct


In [49]:
def evaluate_with_failure_analysis(test_cases):

    correct = 0
    total = len(test_cases)

    failure_log.clear()

    for test in test_cases:

        query = test["query"]
        expected = test["expected_sql"]

        predicted = safe_generate_sql_langchain(query)

        if predicted and predicted.strip().lower() == expected.strip().lower():
            correct += 1

        else:

            failure_type = classify_failure(
                expected,
                predicted
            )

            log_failure(
                query,
                expected,
                predicted,
                failure_type
            )

    accuracy = correct / total

    print(f"\nAccuracy: {accuracy*100:.2f}%")

    generate_report()

In [50]:
evaluate_with_failure_analysis(test_cases)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transforme


Accuracy: 100.00%

===== Evaluation Report =====
Total Failures: 0

Failure Breakdown:


In [44]:
log_failure(
    "Show all users",
    "SELECT * FROM users;",
    "SELECT * FROM orders;",
    "wrong_table"
)

print(failure_log)

[{'query': 'Show all users', 'expected': 'SELECT * FROM users;', 'predicted': 'SELECT * FROM orders;', 'reason': 'wrong_table'}]


In [45]:
from collections import Counter

def failure_summary():

    failure_types = [
        failure["reason"]
        for failure in failure_log
    ]

    summary = Counter(failure_types)

    return summary

In [46]:
failure_log.clear()

log_failure(
    "Query1",
    "SELECT * FROM users;",
    "SELECT * FROM orders;",
    "wrong_table"
)

log_failure(
    "Query2",
    "JOIN query",
    "SELECT * FROM orders;",
    "missing_join"
)

log_failure(
    "Query3",
    "Expected",
    None,
    "generation_failure"
)

print(failure_summary())

Counter({'wrong_table': 1, 'missing_join': 1, 'generation_failure': 1})


In [47]:
def generate_report():

    summary = failure_summary()

    print("\n===== Evaluation Report =====")

    print(f"Total Failures: {len(failure_log)}")

    print("\nFailure Breakdown:")

    for failure_type, count in summary.items():
        print(f"{failure_type}: {count}")

In [48]:
generate_report()


===== Evaluation Report =====
Total Failures: 3

Failure Breakdown:
wrong_table: 1
missing_join: 1
generation_failure: 1


In [51]:
def execution_success(sql):

    try:
        run_sql(sql)
        return True

    except Exception:
        return False

In [52]:
print(
    execution_success(
        "SELECT * FROM users;"
    )
)

print(
    execution_success(
        "SELECT * FROM nonexistent_table;"
    )
)

True
False


In [53]:
def evaluate_with_failure_analysis(test_cases):

    correct = 0
    total = len(test_cases)

    execution_success_count = 0

    failure_log.clear()

    for test in test_cases:

        query = test["query"]
        expected = test["expected_sql"]

        predicted = safe_generate_sql_langchain(query)

        # Exact Match
        if predicted and predicted.strip().lower() == expected.strip().lower():
            correct += 1

        else:

            failure_type = classify_failure(
                expected,
                predicted
            )

            log_failure(
                query,
                expected,
                predicted,
                failure_type
            )

        # Execution Success
        if execution_success(predicted):
            execution_success_count += 1

    accuracy = correct / total
    execution_rate = execution_success_count / total

    print(f"\nAccuracy: {accuracy*100:.2f}%")
    print(f"Execution Success Rate: {execution_rate*100:.2f}%")

    generate_report()

In [54]:
evaluate_with_failure_analysis(test_cases)

[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transforme


Accuracy: 100.00%
Execution Success Rate: 100.00%

===== Evaluation Report =====
Total Failures: 0

Failure Breakdown:
